# Pure DDPG Portfolio Workflow

This notebook is the replacement for `main_code.ipynb`.

It keeps the same core idea, but the code is split into short workflow cells.


## 0. Import shared workflow API

- Keep notebook cells small.
- Put reusable functions in `src/finance_rl_slm/`.
- Use the same helper API as the script version.


In [5]:
from __future__ import annotations

import sys
from stable_baselines3 import DDPG
from dataclasses import replace
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "main":
    PROJECT_ROOT = PROJECT_ROOT.parent

for path in (PROJECT_ROOT / "src", PROJECT_ROOT, PROJECT_ROOT / "src" / "FinRL"):
    path_str = str(path)
    if path.exists() and path_str not in sys.path:
        sys.path.insert(0, path_str)

from finance_rl_slm.config import DEFAULT_CONFIG
from finance_rl_slm.workflow import (
    build_daily_sentiment,
    build_sentiment_inputs,
    load_online_price_data,
    load_price_data,
    plot_normalized_prices,
    print_runtime_context,
    result_picture_path,
    result_profile_path,
    split_price_data,
)
from finance_rl_slm.evaluation import run_online_evaluation
from finance_rl_slm.training import train_offline_model


## 1. Configure the experiment

- Online test window: `2026-01-01` to `2026-06-21`.
- Pipeline name: `only_ddpg`.
- Sentiment input: disabled.


In [6]:
config = replace(
    DEFAULT_CONFIG,
    online_start="2026-01-01",
    online_end="2026-06-21",
)
print_runtime_context(config)


PROJECT_ROOT: /home/tonywang/114-2/PY_Finance_RL_Project
FINRL_ROOT: /home/tonywang/114-2/PY_Finance_RL_Project/src/FinRL
RSS_URLS: {'IBM': 'http://finance.yahoo.com/rss/headline?s=IBM', 'NVDA': 'http://finance.yahoo.com/rss/headline?s=NVDA', 'GM': 'http://finance.yahoo.com/rss/headline?s=GM', 'BLK': 'http://finance.yahoo.com/rss/headline?s=BLK', 'COST': 'http://finance.yahoo.com/rss/headline?s=COST'}


## 2. Download price data and train the offline DDPG model

- This stage learns from price and technical features only.
- The model is saved to `ddpg_portfolio_offline`.
- Running this cell can take a long time.


In [7]:
# The HMI choose list
print("Choose mode \n")
print("1)  Trainnig new DDPG model \n")
print("2)  Loading existing DDPG model \n ")

choice :int = int(input("Enter 1 or 2: ").strip()) 

if choice == 1:
    price_df = load_price_data(config)
    #Stock context check 
    print("price_df range:" , price_df.index.min(),"->", price_df.index.max())
    print("price_df head:\n", price_df.head())
    print("price_df tail:\n", price_df.tail())

    plot_normalized_prices(price_df, config)
    splits = split_price_data(price_df, config)
    model = train_offline_model(splits.train, splits.valid, config)

elif choice == 2:
    model_path = "ddpg_portfolio_offline"
    model = DDPG.load(model_path)
    print(f"Loaded existing model from {model_path}.zip \n")

else:
    raise ValueError("Invalid choice. Please enter 1 or 2.\n")


Choose mode 

1)  Trainnig new DDPG model 

2)  Loading existing DDPG model 
 


ValueError: invalid literal for int() with base 10: ''

## 3. Run online evaluation without SLM

- The environment still has the same observation size.
- The sentiment slot stays `0.0`.
- Plots and profile CSV files are saved under `addenda/`.


In [ ]:
price_df_online = load_online_price_data(config.online_end, config)

only_ddpg_logs = run_online_evaluation(
    price_df_online,
    sentiment_series=None,
    online_end_str=config.online_end,
    config=config,
    save_plots=True,
    plot_dir=result_picture_path(config),
    save_profile=True,
    profile_dir=result_profile_path(config),
    profile_name="only_ddpg",
)

only_ddpg_logs.head()


YF deprecation warning: set proxy via new config function: yf.set_config(proxy=proxy)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Shape of DataFrame:  (580, 8)
price_df_online range: 2026-01-02 00:00:00 -> 2026-06-18 00:00:00
price_df_online shape: (116, 5)
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
df_logs range: 2026-01-05 00:00:00 -> 2026-06-18 00:00:00
              wealth    reward  drawdown  \
time                                       
2026-01-05  1.022432  0.022432  0.000000   
2026-01-06  1.026327  0.003809  0.000000   
2026-01-07  1.016024 -0.020077  0.010038   
2026-01-08  1.024281  0.006134  0.001993   
2026-01-09  1.018930 -0.012431  0.007207   

                                                       action  daily_return  
time                                                                         
2026-01-05  [0.9968072, 0.039360136, 0.7819553, 0.8491447,...           NaN  
2026-01-06  [0.18769988, 0.13324863, 0.13421488, 0.0821841...      0.003809  
2026-01-07  [0.4309695, 0.98682, 0.957307, 0.99957645, 0.3...     -0.010038  
2026-01-08  [0.66359156, 0.75490224, 0

/home/tonywang/114-2/PY_Finance_RL_Project/src/finance_rl_slm/evaluation.py:211: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,wealth,reward,drawdown,action,daily_return
time,,,,,
2026-01-05,1.022432,0.022432,0.000000,"[0.9968072, 0.039360136, 0.7819553, 0.8491447,...",NaN
2026-01-06,1.026327,0.003809,0.000000,"[0.18769988, 0.13324863, 0.13421488, 0.0821841...",0.003809
2026-01-07,1.016024,-0.020077,0.010038,"[0.4309695, 0.98682, 0.957307, 0.99957645, 0.3...",-0.010038
2026-01-08,1.024281,0.006134,0.001993,"[0.66359156, 0.75490224, 0.35768676, 0.7790125...",0.008127
2026-01-09,1.018930,-0.012431,0.007207,"[0.13288537, 0.9835404, 0.99263036, 0.04727947...",-0.005224


## 4. Output notes

- Figure files:
  - `addenda/result_picture/online_reward_only_ddpg_2026-01-01_2026-06-21.png`
  - `addenda/result_picture/online_wealth_only_ddpg_2026-01-01_2026-06-21.png`
  - `addenda/result_picture/online_daily_return_only_ddpg_2026-01-01_2026-06-21.png`

- Profile file:
  - `addenda/result_profile_comparse/only_ddpg_online_profile_2026-01-01_2026-06-21.csv`

- Use this profile as the baseline when comparing against DDPG+SLM.


## Workflow Notes - Pure DDPG

- The main agent is a price-based DDPG policy.

- The training environment uses:
  - recent portfolio returns,
  - MACD-like wealth trend,
  - Bollinger-style wealth deviation,
  - normalized wealth.

- The online evaluation uses the same trained model.

- No language model is used in this pipeline.

- The goal is to understand the baseline behavior first:
  - Does wealth grow or shrink?
  - Is reward stable or noisy?
  - How large is the drawdown?

- After this notebook runs, run the SLM notebook and compare the two profile CSV files.
